# 📖 Lab 3: Book Tickets

Our third functional requirement: **Users should be able to book tickets to events.**

This is where things get interesting. Unlike viewing and searching (which are read-only), booking is a **write operation** that requires **strong consistency**. The #1 rule: **no double bookings** — two users must never pay for the same seat.

## 🏗️ Architecture — Before (Labs 1-2)

```
┌────────┐         ┌─────────────┐         ┌────────────────┐        ┌──────────────┐
│ Client │────────>│ API Gateway │────────> │ Search Service │───────>│  PostgreSQL  │
│        │────────>│             │────────> │ Event Service  │───────>│              │
└────────┘         └─────────────┘         └────────────────┘        └──────────────┘
```

## 🏗️ Architecture — After (adding Booking)

```
┌────────┐         ┌─────────────┐         ┌────────────────┐        ┌──────────────┐
│        │────────>│             │────────> │ Search Service │───────>│              │
│        │         │             │         ├────────────────┤        │              │
│ Client │────────>│ API Gateway │────────> │ Event Service  │───────>│  PostgreSQL  │
│        │         │             │         ├────────────────┤        │  + tickets   │
│        │────────>│             │─book───> │Booking Service │──TX──> │  + bookings  │
└────────┘         └─────────────┘         │       │        │        └──────────────┘
                                           │       v        │
                    POST /bookings          │    Stripe      │
                    {ticketIds, payment}     │  (payment)     │
                                           └────────────────┘
```

The Booking Service uses **database transactions** to ensure no double bookings. Multiple services share the same database — this is OK when data is tightly coupled.

## Learning Objectives

- Understand why booking requires transactions (ACID)
- Build a simple booking flow with PostgreSQL transactions
- See what happens when two users try to book the same ticket (race condition!)
- Prevent double bookings with row-level locking (`SELECT ... FOR UPDATE`)
- Understand the known UX issue (and preview the fix: temporary reservations)

## 🛠️ Setup

Make sure PostgreSQL is running from Lab 1:

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

Select the **"Ticketmaster (Python)"** kernel (top-right of the notebook).

In [1]:
import psycopg2
import psycopg2.extras
import threading
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM tickets WHERE status = 'available'")
print(f"✅ Connected! {cur.fetchone()[0]} available tickets in the database.")
cur.close()
conn.close()

✅ Connected! 498 available tickets in the database.


## 📊 Current Ticket State

Let's look at the tickets for Event 1 (The Eras Tour - NYC) before we start booking. We'll pick some specific seats to work with throughout this lab.

In [5]:
# Let's see the FLOOR section tickets for Event 1
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT id, section, row_label, seat_number, price, status
    FROM tickets
    WHERE event_id = 1 AND section = 'FLOOR'
    ORDER BY row_label, seat_number
""")
tickets = cur.fetchall()

print("🎫 FLOOR section tickets for 'The Eras Tour - NYC':\n")
print(f"{'ID':<6} {'Section':<8} {'Row':<5} {'Seat':<6} {'Price':<10} {'Status'}")
print("-" * 50)
for t in tickets:
    status_icon = "✅" if t["status"] == "available" else "❌"
    print(f"{t['id']:<6} {t['section']:<8} {t['row_label']:<5} {t['seat_number']:<6} ${t['price']:<9} {status_icon} {t['status']}")

cur.close()
conn.close()

🎫 FLOOR section tickets for 'The Eras Tour - NYC':

ID     Section  Row   Seat   Price      Status
--------------------------------------------------
1      FLOOR    A     1      $250.00    ❌ booked
2      FLOOR    A     2      $250.00    ✅ available
3      FLOOR    A     3      $250.00    ✅ available
4      FLOOR    A     4      $250.00    ✅ available
5      FLOOR    A     5      $250.00    ✅ available
6      FLOOR    A     6      $250.00    ❌ sold
7      FLOOR    A     7      $250.00    ✅ available
8      FLOOR    A     8      $250.00    ✅ available
9      FLOOR    A     9      $250.00    ❌ sold
10     FLOOR    A     10     $250.00    ✅ available
11     FLOOR    B     1      $250.00    ✅ available
12     FLOOR    B     2      $250.00    ✅ available
13     FLOOR    B     3      $250.00    ✅ available
14     FLOOR    B     4      $250.00    ✅ available
15     FLOOR    B     5      $250.00    ✅ available
16     FLOOR    B     6      $250.00    ✅ available
17     FLOOR    B     7      $2

## 🔧 Approach 1: Simple Booking (No Locking)

Let's start with the most naive approach — just check if tickets are available, then update them. This is **intentionally broken** to show why we need transactions with proper locking.

The flow:
1. Read ticket status (is it `available`?)
2. If yes, update status to `booked`
3. Create a booking record

What could go wrong? 🤔

In [3]:
def book_tickets_naive(user_id: int, ticket_ids: list[int]) -> dict:
    """
    ❌ BROKEN: Naive booking — no locking, vulnerable to race conditions.
    
    This is what happens when you just check-then-update without protection.
    """
    conn = get_connection()
    # autocommit=False is the default — each statement is in a transaction
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Step 1: Check if tickets are available
        cur.execute("""
            SELECT id, status, price FROM tickets
            WHERE id = ANY(%s)
        """, (ticket_ids,))
        tickets = cur.fetchall()

        # Verify all tickets exist and are available
        for t in tickets:
            if t["status"] != "available":
                conn.rollback()
                return {"success": False, "error": f"Ticket {t['id']} is already {t['status']}"}

        # ⚠️ DANGER ZONE: between the SELECT above and UPDATE below,
        # another user could book the same ticket!
        # Let's add a small delay to simulate real-world latency
        # and make the race condition more likely to occur.
        time.sleep(0.1)

        # Step 2: Update ticket status
        total_price = sum(float(t["price"]) for t in tickets)
        cur.execute("""
            UPDATE tickets SET status = 'booked'
            WHERE id = ANY(%s)
        """, (ticket_ids,))

        # Step 3: Create booking record
        cur.execute("""
            INSERT INTO bookings (user_id, event_id, total_price, status)
            VALUES (%s, (SELECT event_id FROM tickets WHERE id = %s), %s, 'confirmed')
            RETURNING id
        """, (user_id, ticket_ids[0], total_price))
        booking_id = cur.fetchone()["id"]

        # Step 4: Link tickets to booking
        for tid in ticket_ids:
            cur.execute("""
                INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s)
            """, (booking_id, tid))

        conn.commit()
        return {"success": True, "bookingId": booking_id, "totalPrice": total_price}

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": str(e)}
    finally:
        cur.close()
        conn.close()

print("✅ book_tickets_naive() defined — intentionally broken!")

✅ book_tickets_naive() defined — intentionally broken!


In [4]:
# First, let's pick an available ticket to work with
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id, section, row_label, seat_number, price
    FROM tickets
    WHERE event_id = 1 AND status = 'available'
    ORDER BY id
    LIMIT 1
""")
target_ticket = cur.fetchone()
cur.close()
conn.close()

print(f"🎯 Target ticket: ID={target_ticket['id']}, "
      f"Seat={target_ticket['section']}-{target_ticket['row_label']}-{target_ticket['seat_number']}, "
      f"Price=${target_ticket['price']}")

# Single user booking — this works fine
result = book_tickets_naive(user_id=1, ticket_ids=[target_ticket["id"]])
print(f"\n📗 User 1 booking result: {result}")

🎯 Target ticket: ID=1, Seat=FLOOR-A-1, Price=$250.00

📗 User 1 booking result: {'success': True, 'bookingId': 1, 'totalPrice': 250.0}


## 💥 The Race Condition: Double Booking!

Now let's simulate what happens when **two users try to book the same ticket at the same time**. This is the classic race condition:

```
Time    User A                          User B
────    ──────                          ──────
  1     SELECT → ticket is available
  2                                     SELECT → ticket is available  
  3     UPDATE → set status = 'booked'
  4                                     UPDATE → set status = 'booked'  ← DOUBLE BOOKING!
  5     INSERT booking ✅
  6                                     INSERT booking ✅  ← Both users think they got the ticket
```

Both users read the ticket as "available" before either one updates it. Let's prove this happens.

In [ ]:
# Pick a fresh available ticket for the race condition demo
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id FROM tickets
    WHERE event_id = 1 AND status = 'available'
    ORDER BY id LIMIT 1
""")
race_ticket = cur.fetchone()
cur.close()
conn.close()

race_ticket_id = race_ticket["id"]
print(f"🎯 Both users will try to book ticket ID: {race_ticket_id}\n")

# Simulate two users booking the same ticket concurrently
results = {}

def user_books(user_id: int):
    result = book_tickets_naive(user_id=user_id, ticket_ids=[race_ticket_id])
    results[user_id] = result

# Launch both "users" at the same time
thread_a = threading.Thread(target=user_books, args=(100,))
thread_b = threading.Thread(target=user_books, args=(200,))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("Results:")
for user_id, result in sorted(results.items()):
    status = "✅ SUCCESS" if result["success"] else "❌ FAILED"
    print(f"  User {user_id}: {status} — {result}")

# Check how many bookings exist for this ticket
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM booking_tickets WHERE ticket_id = %s", (race_ticket_id,))
booking_count = cur.fetchone()[0]
cur.close()
conn.close()

if booking_count > 1:
    print(f"\n🚨 DOUBLE BOOKING! Ticket {race_ticket_id} was booked {booking_count} times!")
    print("   Two users both paid for the same seat. This is exactly what we need to prevent.")
else:
    print(f"\n🎲 Got lucky this time — no double booking. But it CAN happen!")
    print("   The race condition is non-deterministic. In production with thousands")
    print("   of concurrent users, it WILL happen.")

## 🔒 Approach 2: Transactions with Row-Level Locking (SELECT ... FOR UPDATE)

The fix: use `SELECT ... FOR UPDATE` inside a transaction. This tells PostgreSQL:  
*"Lock these rows — no one else can read or modify them until my transaction finishes."*

```
Time    User A                              User B
────    ──────                              ──────
  1     BEGIN TRANSACTION
  2     SELECT ... FOR UPDATE → locks row
  3     ticket is available ✅
  4                                         BEGIN TRANSACTION
  5                                         SELECT ... FOR UPDATE → ⏳ BLOCKED (waiting for lock)
  6     UPDATE status = 'booked'
  7     INSERT booking
  8     COMMIT → lock released
  9                                         SELECT returns → status is 'booked' now!
 10                                         ❌ Ticket unavailable → ROLLBACK
```

User B's `SELECT FOR UPDATE` **waits** until User A's transaction commits. By then, the ticket status has changed to `booked`, so User B sees it's no longer available.

In [ ]:
def book_tickets_safe(user_id: int, ticket_ids: list[int]) -> dict:
    """
    ✅ SAFE: Uses SELECT ... FOR UPDATE to lock tickets during the transaction.
    Prevents double bookings by ensuring only one transaction can modify a ticket at a time.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Step 1: Lock and check tickets in a single atomic operation
        # FOR UPDATE = acquire row-level locks, blocking other transactions
        cur.execute("""
            SELECT id, status, price FROM tickets
            WHERE id = ANY(%s)
            FOR UPDATE
        """, (ticket_ids,))
        tickets = cur.fetchall()

        # Check all tickets exist
        if len(tickets) != len(ticket_ids):
            conn.rollback()
            return {"success": False, "error": "One or more tickets not found"}

        # Check all tickets are available (while we hold the lock!)
        for t in tickets:
            if t["status"] != "available":
                conn.rollback()
                return {"success": False, "error": f"Ticket {t['id']} is already {t['status']}"}

        # Simulate some processing time (payment, etc.)
        time.sleep(0.1)

        # Step 2: Update ticket status — safe because we hold the lock
        total_price = sum(float(t["price"]) for t in tickets)
        cur.execute("""
            UPDATE tickets SET status = 'booked'
            WHERE id = ANY(%s)
        """, (ticket_ids,))

        # Step 3: Create booking record
        cur.execute("""
            INSERT INTO bookings (user_id, event_id, total_price, status)
            VALUES (%s, (SELECT event_id FROM tickets WHERE id = %s), %s, 'confirmed')
            RETURNING id
        """, (user_id, ticket_ids[0], total_price))
        booking_id = cur.fetchone()["id"]

        # Step 4: Link tickets to booking
        for tid in ticket_ids:
            cur.execute("""
                INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s)
            """, (booking_id, tid))

        conn.commit()
        return {"success": True, "bookingId": booking_id, "totalPrice": total_price}

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": str(e)}
    finally:
        cur.close()
        conn.close()

print("✅ book_tickets_safe() defined — with SELECT ... FOR UPDATE!")

## 🧪 Testing: Can Two Users Double-Book Now?

Let's run the exact same race condition test, but using the safe version with locking.

In [ ]:
# Pick a fresh available ticket
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id FROM tickets
    WHERE event_id = 1 AND status = 'available'
    ORDER BY id LIMIT 1
""")
safe_ticket = cur.fetchone()
cur.close()
conn.close()

safe_ticket_id = safe_ticket["id"]
print(f"🎯 Both users will try to book ticket ID: {safe_ticket_id}\n")

# Same race condition test — but with locking
results = {}

def user_books_safe(user_id: int):
    result = book_tickets_safe(user_id=user_id, ticket_ids=[safe_ticket_id])
    results[user_id] = result

thread_a = threading.Thread(target=user_books_safe, args=(300,))
thread_b = threading.Thread(target=user_books_safe, args=(400,))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("Results:")
for user_id, result in sorted(results.items()):
    status = "✅ SUCCESS" if result["success"] else "❌ FAILED"
    print(f"  User {user_id}: {status} — {result}")

# Verify only one booking
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM booking_tickets WHERE ticket_id = %s", (safe_ticket_id,))
booking_count = cur.fetchone()[0]
cur.close()
conn.close()

print(f"\n📊 Bookings for ticket {safe_ticket_id}: {booking_count}")
if booking_count == 1:
    print("✅ No double booking! SELECT ... FOR UPDATE prevented the race condition.")
else:
    print("🚨 Something went wrong — this shouldn't happen with FOR UPDATE.")

## 🎟️ Multi-Ticket Booking

In real life, users often buy multiple tickets at once (e.g., 2 seats for a couple). Let's test booking multiple tickets in a single transaction.

In [ ]:
# Get 3 available tickets next to each other
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id, section, row_label, seat_number, price
    FROM tickets
    WHERE event_id = 1 AND status = 'available'
    ORDER BY section, row_label, seat_number
    LIMIT 3
""")
multi_tickets = cur.fetchall()
cur.close()
conn.close()

print("🎫 Booking 3 seats together:\n")
multi_ids = []
for t in multi_tickets:
    print(f"   Ticket {t['id']}: {t['section']}-{t['row_label']}-Seat {t['seat_number']} (${t['price']})")
    multi_ids.append(t["id"])

result = book_tickets_safe(user_id=500, ticket_ids=multi_ids)
print(f"\n📗 Result: {result}")

if result["success"]:
    print(f"\n✅ Booked 3 tickets under booking #{result['bookingId']}")
    print(f"   Total: ${result['totalPrice']}")

## 🧹 Cleanup: Reset Tickets

Let's reset the tickets back to `available` and clear bookings so the data is clean for future labs.

In [ ]:
# Reset all tickets booked during this lab back to available
conn = get_connection()
cur = conn.cursor()

cur.execute("DELETE FROM booking_tickets")
cur.execute("DELETE FROM bookings")
cur.execute("UPDATE tickets SET status = 'available' WHERE status = 'booked'")
conn.commit()

cur.execute("SELECT COUNT(*) FROM tickets WHERE status = 'available'")
print(f"✅ Reset complete. {cur.fetchone()[0]} tickets back to available.")

cur.close()
conn.close()

## 🤔 What's Still Wrong?

We solved double bookings with `SELECT ... FOR UPDATE`. But there's a **fundamental UX problem** in our design:

### The "I typed my credit card for nothing" problem

```
User A                                  User B
──────                                  ──────
Sees seat 42 as "available" ✅
Starts typing credit card info...
                                        Sees seat 42 as "available" ✅
                                        Books seat 42 instantly
                                        ✅ Confirmed!
Submits payment...
❌ "Sorry, seat 42 is no longer available"
😡 Wasted 2 minutes typing payment details!
```

User A spent time filling in payment details for a ticket that was already gone. This is terrible UX, especially during a hot event with 10 million users.

### The Fix (Deep Dive): Temporary Reservations

The solution is to split booking into **two phases**:

```
POST /bookings/reserve   →  Temporarily hold tickets (e.g., 10 minutes)
POST /bookings/confirm   →  Complete payment and finalize booking
```

When a user selects seats, we **reserve** them — marking them as `reserved` with an expiration time. Other users see them as unavailable. The user then has a time window to complete payment. If they don't pay in time, the reservation expires and the tickets become available again.

This is exactly what Ticketmaster does — that countdown timer you see!

> We'll implement this in a deep dive lab. For now, our simple flow satisfies the functional requirement: users CAN book tickets, and we prevent double bookings.

## ✅ Summary

| What We Built | Status |
|---------------|--------|
| Naive booking (check-then-update) | ❌ Vulnerable to double bookings |
| Safe booking (`SELECT ... FOR UPDATE`) | ✅ Prevents double bookings with row locking |
| Multi-ticket booking | ✅ All-or-nothing in a single transaction |

**Key concepts:**
- **ACID transactions** ensure atomicity — either all tickets are booked or none
- **`SELECT ... FOR UPDATE`** acquires row-level locks, serializing concurrent access
- **Race conditions** are non-deterministic — they may not happen every time, but they WILL happen at scale
- **Shared database** across services is OK when data is tightly coupled and you need cross-table transactions

**Known limitation:** Users can lose tickets while filling in payment details → fix with reservation system (deep dive)

**Pattern reference:** See `patterns/dealing-with-contention/` for a deeper exploration of concurrency control patterns